# Objetivo 4 — Entrenamiento, Evaluacion y Analisis Estadistico
Entrena y evalua **todos los modelos** bajo las mismas condiciones experimentales.  
Metricas: accuracy, F1-macro, sensibilidad, especificidad, AUC, inf/img, params.  
Estadistica: ANOVA + t-Student (α=0.05), IC 95% via 5-fold CV sobre test.
**Modelos:** ResNet-18 · HQC-CNN · PEQML · HQCINN-shallow · HQCINN-deep

## 0 · Imports y configuracion

In [1]:
import os, random, time, json
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm.notebook import tqdm
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score,
    recall_score, precision_score
)
from scipy import stats

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


Device: cpu


## 1 · Importar modelos definidos en Obj 2 y Obj 3
> Ejecuta primero `modelos_hqcnn.ipynb` y `objetivo3_resnet18.ipynb` en el mismo kernel,  
> o copia aqui las clases `HQCCNN`, `PEQML`, `HQCINN` y `build_resnet18`.  
> La funcion `build_model` unifica todos los modelos.

In [2]:
# ── Si corres este notebook independiente, importa los modelos asi: ────────
# %run modelos_hqcnn.ipynb
# %run objetivo3_resnet18.ipynb

# ── Factory unificada ────────────────────────────────────────────────────────
from torchvision.models import resnet18, ResNet18_Weights

N_QUANTUM_DIM = 4

def build_resnet18(n_classes):
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Sequential(
        nn.Linear(m.fc.in_features, N_QUANTUM_DIM), nn.ReLU(),
        nn.Linear(N_QUANTUM_DIM, n_classes)
    )
    return m

def build_model(name, n_classes):
    if name == 'resnet18':          return build_resnet18(n_classes)
    elif name == 'hqccnn':          return HQCCNN(n_classes)
    elif name == 'peqml':           return PEQML(n_classes)
    elif name == 'hqcinn_shallow':  return HQCINN(n_classes, variant='shallow')
    elif name == 'hqcinn_deep':     return HQCINN(n_classes, variant='deep')
    else: raise ValueError(name)

MODEL_NAMES = ['resnet18', 'hqccnn', 'peqml', 'hqcinn_shallow', 'hqcinn_deep']
print('Modelos disponibles:', MODEL_NAMES)


Modelos disponibles: ['resnet18', 'hqccnn', 'peqml', 'hqcinn_shallow', 'hqcinn_deep']


## 2 · Rutas, DataLoaders y class weights

In [3]:
BASE_DIR  = Path(os.getcwd())
CHEST_OUT = BASE_DIR / 'etl_output' / 'chest_xray'
LUNG_OUT  = BASE_DIR / 'etl_output' / 'lung_cancer'
CKPT_DIR  = BASE_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
BATCH_SIZE = 32
IMG_SIZE   = (128, 128)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_train = T.Compose([
    T.Resize(IMG_SIZE), T.Grayscale(num_output_channels=3),
    T.RandomRotation(10), T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.15),
    T.RandomAffine(degrees=0, scale=(0.90, 1.10)),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
transform_eval = T.Compose([
    T.Resize(IMG_SIZE), T.Grayscale(num_output_channels=3),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def make_loaders(root, weighted=False):
    root = Path(root)
    ds_tr = ImageFolder(root/'train', transform=transform_train)
    ds_va = ImageFolder(root/'val',   transform=transform_eval)
    ds_te = ImageFolder(root/'test',  transform=transform_eval)
    if weighted:
        tgts = torch.tensor(ds_tr.targets)
        sw   = (1.0/torch.bincount(tgts).float())[tgts]
        ltr  = DataLoader(ds_tr, batch_size=BATCH_SIZE,
                          sampler=WeightedRandomSampler(sw,len(sw),True),
                          num_workers=2, pin_memory=True)
    else:
        ltr  = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
    lva = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    lte = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return ltr, lva, lte, ds_tr.class_to_idx

chest_train_l, chest_val_l, chest_test_l, chest_cls = make_loaders(CHEST_OUT)
lung_train_l,  lung_val_l,  lung_test_l,  lung_cls  = make_loaders(LUNG_OUT, weighted=True)
N_CHEST = len(chest_cls)
N_LUNG  = len(lung_cls)

tgts = torch.tensor(ImageFolder(LUNG_OUT/'train').targets)
lung_cw = (1.0/torch.bincount(tgts).float())
lung_cw = (lung_cw/lung_cw.sum()).to(DEVICE)

CHEST_NAMES = ['NORMAL', 'PNEUMONIA']
LUNG_NAMES  = ['Benign', 'Malignant', 'Normal']
print('Chest:', chest_cls, '| Lung:', lung_cls)


Chest: {'NORMAL': 0, 'PNEUMONIA': 1} | Lung: {'Benign': 0, 'Malignant': 1, 'Normal': 2}


## 3 · Loop de entrenamiento

In [4]:
def train_one_model(model, train_loader, val_loader, class_weights=None,
                    max_epochs=50, patience=5, save_path=None):
    """
    Protocolo unificado para todos los modelos (PPI Fase 3):
    Adam lr=1e-3 -> 1e-4 (epoch 10), early stopping patience=5 sobre F1-macro.
    """
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10], gamma=0.1)
    hist = {'train_loss':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}
    best_f1, no_imp = 0.0, 0

    for ep in range(1, max_epochs+1):
        model.train()
        tloss = 0.0
        for imgs, lbls in tqdm(train_loader, desc=f'Ep {ep}/{max_epochs}', leave=False):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward(); optimizer.step()
            tloss += loss.item() * imgs.size(0)
        scheduler.step()
        avg_tr = tloss / len(train_loader.dataset)

        model.eval()
        vloss, preds, labs = 0.0, [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                logits = model(imgs)
                vloss += criterion(logits, lbls).item() * imgs.size(0)
                preds.extend(logits.argmax(1).cpu().numpy())
                labs.extend(lbls.cpu().numpy())
        avg_va  = vloss / len(val_loader.dataset)
        val_acc = sum(p==l for p,l in zip(preds,labs)) / len(labs)
        val_f1  = f1_score(labs, preds, average='macro', zero_division=0)
        hist['train_loss'].append(avg_tr); hist['val_loss'].append(avg_va)
        hist['val_acc'].append(val_acc);   hist['val_f1'].append(val_f1)
        print(f'Ep {ep:3d} | train={avg_tr:.4f} | val={avg_va:.4f} | acc={val_acc:.4f} | F1={val_f1:.4f}')

        if val_f1 > best_f1:
            best_f1, no_imp = val_f1, 0
            if save_path: torch.save(model.state_dict(), save_path)
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'Early stopping ep {ep}'); break
    return hist


## 4 · Metricas estandarizadas

In [5]:
def compute_metrics(model, test_loader, class_names, model_name):
    """
    Calcula: accuracy, F1-macro, sensibilidad (recall macro), especificidad,
    AUC, tiempo de inferencia por imagen, params entrenables.
    """
    model = model.to(DEVICE).eval()
    preds, labs, probs = [], [], []
    t0 = time.time()
    with torch.no_grad():
        for imgs, lbls in test_loader:
            logits = model(imgs.to(DEVICE))
            probs.extend(torch.softmax(logits,1).cpu().numpy())
            preds.extend(logits.argmax(1).cpu().numpy())
            labs.extend(lbls.numpy())
    inf_ms = (time.time()-t0) / len(labs) * 1000

    n_cls  = len(class_names)
    acc    = accuracy_score(labs, preds)
    f1_mac = f1_score(labs, preds, average='macro', zero_division=0)
    sens   = recall_score(labs, preds, average='macro', zero_division=0)  # sensibilidad = recall

    # Especificidad por clase: TN/(TN+FP), luego macro-promedio
    cm = confusion_matrix(labs, preds)
    spec_per_class = []
    for i in range(n_cls):
        TP = cm[i,i]
        FN = cm[i,:].sum() - TP
        FP = cm[:,i].sum() - TP
        TN = cm.sum() - TP - FN - FP
        spec_per_class.append(TN / (TN+FP) if (TN+FP) > 0 else 0.0)
    spec = float(np.mean(spec_per_class))

    try:
        auc = roc_auc_score(labs, probs, multi_class='ovr', average='macro') \
              if n_cls > 2 else roc_auc_score(labs, [p[1] for p in probs])
    except Exception:
        auc = float('nan')

    params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    result = dict(model=model_name, acc=acc, f1=f1_mac,
                  sens=sens, spec=spec, auc=auc,
                  inf_ms=inf_ms, params=params,
                  preds=preds, labs=labs, probs=probs)

    print(f'\n=== {model_name} ===')
    print(f'  Acc={acc:.4f}  F1={f1_mac:.4f}  Sens={sens:.4f}  Spec={spec:.4f}  AUC={auc:.4f}')
    print(f'  Inf={inf_ms:.3f}ms  Params={params:,}')
    print()
    print(classification_report(labs, preds, target_names=class_names, zero_division=0))
    return result


def plot_confusion(result, class_names):
    cm = confusion_matrix(result['labs'], result['preds'])
    plt.figure(figsize=(max(4,len(class_names)*2), max(3,len(class_names)*1.5)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix — {result["model"]}')
    plt.ylabel('Real'); plt.xlabel('Predicho')
    plt.tight_layout(); plt.show()


def plot_history(hist, title):
    fig, axes = plt.subplots(1,2,figsize=(12,4))
    fig.suptitle(f'Curvas — {title}', fontweight='bold')
    axes[0].plot(hist['train_loss'], label='train')
    axes[0].plot(hist['val_loss'],   label='val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(hist['val_acc'], label='acc')
    axes[1].plot(hist['val_f1'],  label='F1-macro')
    axes[1].set_title('Val Accuracy / F1'); axes[1].legend()
    plt.tight_layout(); plt.show()


## 5 · Entrenamiento de todos los modelos
Ejecuta celda a celda. Los checkpoints se guardan en `checkpoints/`.  
Cambia `DATASET` para alternar entre `chest` y `lung`.

In [ ]:
# ── Configuracion ────────────────────────────────────────────────────────────
DATASET = 'chest'   # 'chest' | 'lung'

if DATASET == 'chest':
    train_l, val_l, test_l = chest_train_l, chest_val_l, chest_test_l
    n_cls, cls_names, cw = N_CHEST, CHEST_NAMES, None
else:
    train_l, val_l, test_l = lung_train_l, lung_val_l, lung_test_l
    n_cls, cls_names, cw = N_LUNG, LUNG_NAMES, lung_cw

save_dir = CKPT_DIR / DATASET
save_dir.mkdir(exist_ok=True)

histories, results = {}, {}

for mname in MODEL_NAMES:
    print(f'\n{'='*55}')
    print(f'Entrenando: {mname} | dataset: {DATASET}')
    print('='*55)
    model = build_model(mname, n_cls).to(DEVICE)
    ckpt  = str(save_dir / f'{mname}_best.pt')

    histories[mname] = train_one_model(
        model, train_l, val_l,
        class_weights=cw, max_epochs=50, patience=5, save_path=ckpt
    )
    plot_history(histories[mname], f'{mname} — {DATASET}')

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    results[mname] = compute_metrics(model, test_l, cls_names, mname)
    plot_confusion(results[mname], cls_names)

print('\nTodos los modelos entrenados y evaluados.')



Entrenando: resnet18 | dataset: chest


Ep 1/50:   0%|          | 0/129 [00:00<?, ?it/s]

D:\CURSOS 2026 - 1\Cyberseguridad\jupyterlab\envprueba\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Ep   1 | train=0.1979 | val=0.1581 | acc=0.9385 | F1=0.9254


Ep 2/50:   0%|          | 0/129 [00:00<?, ?it/s]

...

Ep 5/50:   0%|          | 0/129 [00:00<?, ?it/s]

D:\CURSOS 2026 - 1\Cyberseguridad\jupyterlab\envprueba\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Ep   5 | train=0.1119 | val=0.4593 | acc=0.8246 | F1=0.8086


Ep 6/50:   0%|          | 0/129 [00:00<?, ?it/s]

D:\CURSOS 2026 - 1\Cyberseguridad\jupyterlab\envprueba\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 6 · Tabla comparativa de metricas

In [ ]:
rows = []
for mname, r in results.items():
    rows.append({
        'Modelo': mname,
        'Accuracy': round(r['acc'],4),
        'F1-macro': round(r['f1'],4),
        'Sensibilidad': round(r['sens'],4),
        'Especificidad': round(r['spec'],4),
        'AUC': round(r['auc'],4),
        'Inf(ms)': round(r['inf_ms'],3),
        'Params': r['params'],
    })

df = pd.DataFrame(rows).set_index('Modelo')
print(f'=== Dataset: {DATASET} ===')
print(df.to_string())

# Guardar como CSV
df.to_csv(BASE_DIR / f'resultados_{DATASET}.csv')
print(f'\nGuardado: resultados_{DATASET}.csv')


## 7 · Visualizacion comparativa

In [ ]:
metrics_to_plot = ['Accuracy', 'F1-macro', 'Sensibilidad', 'Especificidad', 'AUC']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(18, 4))
fig.suptitle(f'Comparativa de modelos — {DATASET}', fontweight='bold')

palette = sns.color_palette('Set2', len(df))
for ax, metric in zip(axes, metrics_to_plot):
    vals = df[metric].values
    bars = ax.bar(df.index, vals, color=palette)
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}',
                ha='center', fontsize=7)
plt.tight_layout(); plt.show()


## 8 · Analisis estadistico — ANOVA + t-Student (α = 0.05)
IC 95% via bootstrap sobre las predicciones del conjunto de test.

In [ ]:
def bootstrap_f1(preds, labs, n_boot=1000, seed=42):
    """IC 95% para F1-macro via bootstrap."""
    rng = np.random.default_rng(seed)
    n   = len(labs)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        scores.append(f1_score(
            np.array(labs)[idx], np.array(preds)[idx],
            average='macro', zero_division=0
        ))
    return np.percentile(scores, [2.5, 97.5])


print(f'=== IC 95% F1-macro — {DATASET} ===')
f1_scores = {}
for mname, r in results.items():
    ci = bootstrap_f1(r['preds'], r['labs'])
    f1_scores[mname] = r['f1']
    print(f'  {mname:<20}: F1={r["f1"]:.4f}  IC95=[{ci[0]:.4f}, {ci[1]:.4f}]')


In [ ]:
# ── ANOVA de una via sobre F1 ──────────────────────────────────────────────
# Genera distribucion de F1 via bootstrap para cada modelo
def f1_bootstrap_dist(preds, labs, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n   = len(labs)
    return [
        f1_score(np.array(labs)[rng.integers(0,n,n)],
                 np.array(preds)[rng.integers(0,n,n)],
                 average='macro', zero_division=0)
        for _ in range(n_boot)
    ]

dists = {mn: f1_bootstrap_dist(results[mn]['preds'], results[mn]['labs'])
         for mn in MODEL_NAMES if mn in results}

f_stat, p_anova = stats.f_oneway(*dists.values())
print(f'ANOVA F1-macro — F={f_stat:.4f}  p={p_anova:.6f}')
if p_anova < 0.05:
    print('  → Diferencias estadisticamente significativas (α=0.05)')
else:
    print('  → Sin diferencias significativas (α=0.05)')

print()
print('t-Student pareado (todos los pares):')
alpha = 0.05
pairs = list(combinations(list(dists.keys()), 2))
pair_rows = []
for m1, m2 in pairs:
    t, p = stats.ttest_ind(dists[m1], dists[m2])
    sig  = '*' if p < alpha else ''
    pair_rows.append({'Par': f'{m1} vs {m2}', 't': round(t,4), 'p': round(p,6), 'sig': sig})
    print(f'  {m1} vs {m2}: t={t:.4f}  p={p:.6f}  {sig}')

df_pairs = pd.DataFrame(pair_rows)
df_pairs.to_csv(BASE_DIR / f'ttest_{DATASET}.csv', index=False)
print(f'\nGuardado: ttest_{DATASET}.csv')


## 9 · Guardar resultados consolidados

In [ ]:
summary = {
    'dataset': DATASET,
    'metrics': {mn: {k:v for k,v in r.items() if k not in ('preds','labs','probs')}
                for mn, r in results.items()},
    'anova': {'F': round(float(f_stat),4), 'p': round(float(p_anova),6)},
    'ttest_pairs': pair_rows,
}
out_path = BASE_DIR / f'resultados_completos_{DATASET}.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Guardado: {out_path}')
